# Violent Detection Training Notebook

Notebook này gom các file `config.py`, `utils.py`, `dataset.py`, model và `train.py` lại thành một file `.ipynb` duy nhất.
Cách gộp này giúp bạn dễ dàng chạy và theo dõi quá trình huấn luyện trực tiếp trên môi trường Server (như JupyterLab, VSCode Server, Colab hoặc Kaggle) mà không cần phải chuyển đổi qua lại giữa các file `.py`.

## 1. Import các Thư viện cần thiết

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

## 2. Thiết lập Cấu hình (Config)

In [ ]:
class Config:
    # Đường dẫn (Paths)
    DATA_DIR = "my_dataset"
    PROCESSED_DATA_DIR = "my_dataset/processed_data"

    # Thông số Video
    T_FRAMES = 16              # Số khung hình cho mỗi chuỗi (Sequence Length)
    IMAGE_SIZE = 224           # Kích thước ảnh chuẩn cho EfficientNet
    
    # Thông số Model
    FEATURE_DIM = 1280         # Kích thước vector đầu ra của EfficientNet-B0
    LSTM_HIDDEN_SIZE = 256     # Kích thước hidden state của LSTM
    LSTM_LAYERS = 1            # Số tầng LSTM
    
    # Thông số Huấn luyện (Training)
    BATCH_SIZE = 16            # Giảm xuống 2 hoặc 4 để tránh lỗi CUDA Out of Memory
    LEARNING_RATE = 1e-4
    EPOCHS = 50
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()

## 3. Utils & Dataset

In [ ]:
def compute_farneback_flow(prev_frame, next_frame):
    """
    Tính Dense Optical Flow bằng Farneback.
    """
    prvs = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
    nxt = cv2.cvtColor(next_frame, cv2.COLOR_BGR2GRAY)
    
    flow = cv2.calcOpticalFlowFarneback(prvs, nxt, None, 
                                        0.5, 3, 15, 3, 5, 1.2, 0)
    
    hsv = np.zeros_like(prev_frame)
    hsv[..., 1] = 255
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    
    flow_rgb = cv2.cvtColor(hsv, cv2.COLOR_HSV2RGB)
    return flow_rgb

class PreprocessedVideoDataset(Dataset):
    def __init__(self, data_dir, phase='train'):
        self.data_dir = os.path.join(data_dir, phase)
        self.file_paths = []
        self.labels = []
        
        v_dir = os.path.join(self.data_dir, 'violence')
        if os.path.exists(v_dir):
            for f in os.listdir(v_dir):
                if f.endswith('.pt'):
                    self.file_paths.append(os.path.join(v_dir, f))
                    self.labels.append(1.0)
        
        nv_dir = os.path.join(self.data_dir, 'non_violence')
        if os.path.exists(nv_dir):
            for f in os.listdir(nv_dir):
                if f.endswith('.pt'):
                    self.file_paths.append(os.path.join(nv_dir, f))
                    self.labels.append(0.0)

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        data = torch.load(self.file_paths[idx])
        rgb_seq = data['rgb']     # Shape: (16, 3, 224, 224)
        flow_seq = data['flow']   # Shape: (16, 3, 224, 224)
        label = torch.tensor([self.labels[idx]], dtype=torch.float32)
        return rgb_seq, flow_seq, label

## 4. Kiến trúc Mô hình (Model Architecture)
Được xây dựng lại dựa trên `ValdNetBaseline` với kiến trúc EfficientNet + LSTM

In [ ]:
class ValdNetBaseline(nn.Module):
    def __init__(self, cfg):
        super(ValdNetBaseline, self).__init__()
        # Backbone đặc trưng luồng RGB
        self.backbone_rgb = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.backbone_rgb.classifier = nn.Identity() 
        
        # Backbone đặc trưng luồng Optical Flow
        self.backbone_flow = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
        self.backbone_flow.classifier = nn.Identity()
        
        # LSTM học từ chuỗi kết hợp (RGB + Flow)
        self.lstm = nn.LSTM(input_size=cfg.FEATURE_DIM * 2, 
                            hidden_size=cfg.LSTM_HIDDEN_SIZE, 
                            num_layers=cfg.LSTM_LAYERS, 
                            batch_first=True)
                            
        self.fc = nn.Linear(cfg.LSTM_HIDDEN_SIZE, 1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, rgb_seq, flow_seq):
        batch_size, t, c, h, w = rgb_seq.size()
        
        rgb_seq = rgb_seq.view(batch_size * t, c, h, w)
        flow_seq = flow_seq.view(batch_size * t, c, h, w)
        
        rgb_feat = self.backbone_rgb(rgb_seq)
        flow_feat = self.backbone_flow(flow_seq)
        
        combined_feat = torch.cat((rgb_feat, flow_feat), dim=1)
        combined_feat = combined_feat.view(batch_size, t, -1)
        
        lstm_out, _ = self.lstm(combined_feat)
        last_out = lstm_out[:, -1, :]
        
        out = self.fc(last_out)
        return self.sigmoid(out)

## 5. Vòng lặp Huấn luyện (Training Loop)

In [ ]:
def train_model():
    device = torch.device(cfg.DEVICE)
    model = ValdNetBaseline(cfg).to(device)
    print(f"Bắt đầu huấn luyện trên thiết bị: {device}")
    
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
    
    train_dataset = PreprocessedVideoDataset(data_dir=cfg.PROCESSED_DATA_DIR, phase='train')
    train_loader = DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
    
    val_dataset = PreprocessedVideoDataset(data_dir=cfg.PROCESSED_DATA_DIR, phase='val')
    val_loader = DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)
    
    if len(train_loader) == 0 or len(val_loader) == 0:
        print("LỖI NGHIÊM TRỌNG: DataLoader trống! Vui lòng kiểm tra lại thư mục dữ liệu.")
        return

    best_val_loss = float('inf')
    
    for epoch in range(cfg.EPOCHS):
        # --- HUẤN LUYỆN ---
        model.train()
        total_loss, correct_train, total_train = 0, 0, 0
        
        train_bar = tqdm(train_loader, desc=f"Epoch [{epoch+1}/{cfg.EPOCHS}] [Train]")
        for rgb_seq, flow_seq, labels in train_bar:
            rgb_seq, flow_seq, labels = rgb_seq.to(device), flow_seq.to(device), labels.to(device)
            
            optimizer.zero_grad()
            predictions = model(rgb_seq, flow_seq).view(-1)
            labels = labels.view(-1)
            predictions = torch.clamp(predictions, min=0.0, max=1.0)

            loss = criterion(predictions, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            predicted = (predictions > 0.5).float()
            correct_train += (predicted == labels).sum().item()
            total_train += labels.size(0)
            
            train_bar.set_postfix(loss=loss.item())
            
        train_loss = total_loss / len(train_loader)
        train_acc = correct_train / total_train
        
        # --- ĐÁNH GIÁ ---
        model.eval()
        val_loss, correct_val, total_val = 0, 0, 0
        
        with torch.no_grad():
            val_bar = tqdm(val_loader, desc=f"Epoch [{epoch+1}/{cfg.EPOCHS}] [Val]  ")
            for rgb_seq, flow_seq, labels in val_bar:
                rgb_seq, flow_seq, labels = rgb_seq.to(device), flow_seq.to(device), labels.to(device)
                
                predictions = model(rgb_seq, flow_seq).view(-1)
                labels = labels.view(-1)
                predictions = torch.clamp(predictions, min=0.0, max=1.0)

                loss = criterion(predictions, labels)
                
                val_loss += loss.item()
                predicted = (predictions > 0.5).float()
                correct_val += (predicted == labels).sum().item()
                total_val += labels.size(0)
                
        val_loss /= len(val_loader)
        val_acc = correct_val / total_val
            
        print(f"\n=> Kết quả Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}")
              
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "best_model.pth")
            print("   [+] Đã lưu model tốt nhất (best_model.pth)!\n")

# Chạy huấn luyện
train_model()